In [73]:
import numpy as np
import  pandas as pd 
import networkx as nx
import matplotlib.pyplot as plt
import geopandas as gpd
from geopy.distance import geodesic
import contextily as ctx
from sklearn.preprocessing import MinMaxScaler



### Load data

In [74]:
houses =pd.read_csv("minimal_example/synthetic_data/houses_v2.csv")
regions =pd.read_csv("minimal_example/synthetic_data/regions.csv")
schools =pd.read_csv("minimal_example/synthetic_data/schools_v2.csv")
stations =pd.read_csv("minimal_example/synthetic_data/stations_v2.csv")


In [75]:
houses.head()

,property_id,price,rooms,bathrooms,year_renovated,eco_score,heating_type,exterior_material,lat,lon,surface_total
0,H0,5.247241e+08,3,1,2000,75.0,gas,brick,4.620924,-74.129537,199.035382
1,H1,8.704286e+08,2,1,2000,75.0,gas,brick,4.643822,-74.142194,106.877704
2,H2,7.391964e+08,3,2,2000,75.0,gas,brick,4.654954,-74.036134,281.864080
3,H3,6.591951e+08,5,2,2000,75.0,gas,brick,4.668410,-74.034124,151.755996
4,H4,3.936112e+08,5,1,2000,75.0,gas,brick,4.717776,-74.052992,232.504457


In [76]:
house_ids = houses["property_id"] 
station_ids = stations["poi_id"]
school_ids = schools["poi_id"]

### Add nodes

In [77]:
G = nx.Graph()

In [78]:
for _, row in houses.iterrows():
    G.add_node(
     
     row['property_id'],
     type = 'house',
     price = row['price'],
     rooms = row['rooms'],
     bathrooms = row['bathrooms'],
     surface_total = row['surface_total'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

In [79]:
for _, row in stations.iterrows():
    G.add_node(
     
     row['poi_id'],
     type = 'station',
     daily_traffic = row['daily_traffic'],
     connections = row['connections'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

In [80]:
for _, row in schools.iterrows():
    G.add_node(
     row['poi_id'],
     type = 'school',
     ranking = row['ranking'],
     students = row['students'],
     lat = row['lat'],
     lon = row['lon']
     
        
    )

### Add edges

In [81]:
def add_edges(G, df1, df2):
    df1_id = df1.columns[df1.columns.str.endswith("_id")][0]
    df2_id = df2.columns[df2.columns.str.endswith("_id")][0]
    
    for _, row in df1.iterrows():
        coords_1 = (row['lat'], row['lon'])
        
        for _, rowa in df2.iterrows():
            coords_2 = (rowa['lat'], rowa['lon'])
            
            distance = geodesic(coords_1, coords_2).km
            
            if distance < 5:  # Connect if within 5 km
                
                G.add_edge(
                    row[df1_id], rowa[df2_id], weight= 1/distance)
    return G
            

In [82]:
G = add_edges(G, houses, stations)
G = add_edges(G, houses, schools)
G = add_edges(G, schools, stations)

In [83]:
G.edges()

EdgeView([('H0', 'T2'), ('H0', 'T4'), ('H0', 'S4'), ('H1', 'T2'), ('H1', 'T4'), ('H1', 'S4'), ('H3', 'T3'), ('H3', 'S3'), ('H4', 'T3'), ('H4', 'T5'), ('H4', 'S1'), ('H4', 'S3'), ('H5', 'T2'), ('H5', 'T4'), ('H5', 'S2'), ('H5', 'S4'), ('H6', 'T2'), ('H6', 'T4'), ('H6', 'S2'), ('H6', 'S4'), ('H7', 'T1'), ('H7', 'T3'), ('H7', 'T5'), ('H7', 'S1'), ('H7', 'S2'), ('H7', 'S3'), ('H8', 'T4'), ('T1', 'S1'), ('T1', 'S2'), ('T1', 'S3'), ('T2', 'S1'), ('T2', 'S2'), ('T2', 'S4'), ('T3', 'S1'), ('T3', 'S3'), ('T4', 'S2'), ('T4', 'S4'), ('T5', 'S3')])

### X feature Matrixes

In [84]:
def build_X_features(df, target=None):
    df1 = df.copy()
    
    # 1. Drop ID + lat/lon
    id_cols = [c for c in df1.columns if c.endswith("_id")]
    df1 = df1.drop(columns=id_cols + ["lat", "lon"], errors="ignore")
    
    if target is not None and target in df1.columns:
        df1 = df1.drop(columns=target, errors="ignore")

    # 2. Separate numeric + categorical
    numeric_cols = df1.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df1.select_dtypes(include=["object", "category", "bool"]).columns

    # 3. Impute missing values
    df1[numeric_cols] = df1[numeric_cols].fillna(df1[numeric_cols].median())
    df1[cat_cols] = df1[cat_cols].fillna("Unknown")

    # 4. One-hot encode categoricals
    df1 = pd.get_dummies(df1, columns=cat_cols, drop_first=False)

    # 5. Scale numeric columns (per type!)
    scaler = MinMaxScaler()
    df1[numeric_cols] = scaler.fit_transform(df1[numeric_cols])

    return df1


In [85]:
house_X = build_X_features(houses, 'price') # Remove the targer column
school_X = build_X_features(schools)
station_X = build_X_features(stations)

In [86]:
house_X.head()

,rooms,bathrooms,year_renovated,eco_score,surface_total,heating_type_electric,heating_type_gas,exterior_material_brick,exterior_material_wood
0,0.333333,0.0,0.0,0.0,0.492718,False,True,True,False
1,0.000000,0.0,0.0,0.0,0.000000,False,True,True,False
2,0.333333,0.5,0.0,0.0,0.935560,False,True,True,False
3,1.000000,0.5,0.0,0.0,0.239941,False,True,True,False
4,1.000000,0.0,0.0,0.0,0.671660,False,True,True,False


In [87]:
def clean_bools(df1):
    bool_cols = df1.select_dtypes(include=["bool"]).columns
    df1[bool_cols] = df1[bool_cols].astype(np.float32)
    
    return df1

house_X = clean_bools(house_X)
school_X = clean_bools(school_X)
station_X = clean_bools(station_X)

### Compression

In [88]:
import torch
import torch.nn as nn

In [89]:
house_X.values

array([[0.33333333, 0.        , 0.        , 0.        , 0.49271846,
        0.        , 1.        , 1.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 1.        , 1.        , 0.        ],
       [0.33333333, 0.5       , 0.        , 0.        , 0.9355598 ,
        0.        , 1.        , 1.        , 0.        ],
       [1.        , 0.5       , 0.        , 0.        , 0.23994054,
        0.        , 1.        , 1.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        , 0.67165994,
        0.        , 1.        , 1.        , 0.        ],
       [0.33333333, 0.        , 0.        , 0.        , 0.29653947,
        0.        , 1.        , 1.        , 0.        ],
       [0.33333333, 0.        , 0.        , 0.        , 0.51933439,
        0.        , 1.        , 1.        , 0.        ],
       [0.33333333, 1.        , 0.        , 0.        , 0.54782281,
        0.        , 1.        , 1.        , 0.        ],


In [90]:
# Tensors

X_house = torch.tensor(house_X.values, dtype = torch.float32)
X_school = torch.tensor(school_X.values, dtype = torch.float32)
X_station = torch.tensor(station_X.values, dtype = torch.float32)

In [91]:
class Linear_compressor(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, latent_dim)
    
    def forward(self, x):
        return self.linear(x)


In [92]:
r = 3

house_compressor = Linear_compressor(X_house.shape[1], r)
school_compressor = Linear_compressor(X_school.shape[1], r)
station_compressor = Linear_compressor(X_station.shape[1], r)

house_compressed = house_compressor(X_house)
school_compressed = school_compressor(X_school)
station_compressed = station_compressor(X_station)

In [97]:
def compress_to_df(C, index_list):
    U = C.detach().cpu().numpy()
    
    U_df = pd.DataFrame(
        U,
        index= index_list,
        columns=[f"u{i+1}" for i in range(U.shape[1])]
    )
    return U_df
    
    

In [ ]:
U_house_df = compress_to_df(house_compressed, house_ids)
U_school_df = compress_to_df(school_compressed, school_ids)
U_station_df = compress_to_df(station_compressed, station_ids)

### Gaussian Spaces